In [1]:
import os
import sys
import pickle
import glob
import xarray as xr
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import torch
import matplotlib.pyplot as plt

print(f"Current working directory: {os.getcwd()}")

from neuralhydrology.utils.config import Config
from neuralhydrology.datasetzoo.onlineforecastdataset import OnlineForecastDataset
from neuralhydrology.nh_run import start_run, eval_run
from neuralhydrology.training.train import start_training

Current working directory: /home/sngrj0hn/GitHub/neuralhydrology/operational_harz/gefs_10d_sample


## Load Configuration

In [2]:
config_path = Path("DE_2_basin.yml")
cfg = Config(config_path)
print(f"Loaded config for experiment: {cfg.experiment_name}")
print(f"Run directory: {cfg.run_dir}")

Loaded config for experiment: development_run
Run directory: None


## Inspect Data Loading
This step manually instantiates the `OnlineForecastDataset` to verify that:
1. The cached `train_data.zarr` can be accessed or rebuilt.
2. The scaler is correctly loaded or created.

In [3]:
# Initialize the dataset in training mode
# This will trigger _load_or_create_xarray_dataset and _ensure_scaler
try:
    print("Attempting to load OnlineForecastDataset...")
    ds = OnlineForecastDataset(cfg=cfg, is_train=True, period="train")
    print("Successfully loaded OnlineForecastDataset!")
    print(f"Number of samples: {len(ds)}")
    print(f"Scaler keys: {list(ds.scaler.keys())}")

    # Convert Zarr cache to Pickle for training
    # This allows us to use the cached data while letting the training run manage its own directory structure
    zarr_path = cfg.train_dir / "train_data.zarr"
    pickle_path = Path("runs/train_data.p")
    pickle_path.parent.mkdir(parents=True, exist_ok=True)
    
    if zarr_path.exists():
        print(f"Converting Zarr cache from {zarr_path} to pickle at {pickle_path}...")
        # Load zarr
        ds_xr = xr.open_zarr(zarr_path)
        # Save as pickle
        with open(pickle_path, "wb") as fp:
            pickle.dump(ds_xr.to_dict(), fp)
        print("Conversion complete.")
    else:
        print("Warning: Zarr cache not found, cannot create pickle.")

except Exception as e:
    print(f"Error loading dataset: {e}")
    raise

Attempting to load OnlineForecastDataset...


/home/sngrj0hn/anaconda3/envs/neuralhydrology/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


100%|██████████| 1/1 [00:00<00:00,  2.04it/s]

Successfully loaded OnlineForecastDataset!
Number of samples: 1095
Scaler keys: ['xarray_feature_scale', 'xarray_feature_center']
Converting Zarr cache from runs/train_data.zarr to pickle at runs/train_data.p...
Successfully loaded OnlineForecastDataset!
Number of samples: 1095
Scaler keys: ['xarray_feature_scale', 'xarray_feature_center']
Converting Zarr cache from runs/train_data.zarr to pickle at runs/train_data.p...


/home/sngrj0hn/anaconda3/envs/neuralhydrology/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)
/tmp/ipykernel_1323895/3364880289.py:19: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds_xr = xr.open_zarr(zarr_path)


Conversion complete.


## Train Model
If the data loading above succeeded, we can proceed to train the model.

In [4]:
# Modify config to use the pickled data and default run directory structure
# This ensures the scaler is saved in the correct run-specific folder where validation expects it
# We use update_config because some properties (like train_data_file) do not have setters
cfg.update_config({
    "train_dir": None,
    "train_data_file": Path("runs/train_data.p"),
    "save_train_data": False
})

if torch.cuda.is_available():
    print(f"Starting run on GPU: {torch.cuda.get_device_name(0)}")
    cfg.device = "cuda:0"
else:
    print("Starting run on CPU")
    cfg.device = "cpu"

start_training(cfg)

Starting run on GPU: NVIDIA GeForce RTX 4090
2025-11-21 09:41:51,156: Logging to /home/sngrj0hn/GitHub/neuralhydrology/operational_harz/gefs_10d_sample/runs/development_run_2111_094151/output.log initialized.
2025-11-21 09:41:51,156: ### Folder structure created at /home/sngrj0hn/GitHub/neuralhydrology/operational_harz/gefs_10d_sample/runs/development_run_2111_094151
2025-11-21 09:41:51,157: ### Run configurations for development_run
2025-11-21 09:41:51,157: experiment_name: development_run
2025-11-21 09:41:51,157: run_dir: /home/sngrj0hn/GitHub/neuralhydrology/operational_harz/gefs_10d_sample/runs/development_run_2111_094151
2025-11-21 09:41:51,157: train_basin_file: basins.txt
2025-11-21 09:41:51,157: validation_basin_file: basins.txt
2025-11-21 09:41:51,157: test_basin_file: basins.txt
2025-11-21 09:41:51,158: train_start_date: 2020-10-01 00:00:00
2025-11-21 09:41:51,158: train_end_date: 2023-09-30 00:00:00
2025-11-21 09:41:51,158: validation_start_date: 2023-10-01 00:00:00
2025-11-

## Evaluation
Now that the model is trained, we evaluate it on the test set to generate predictions.

In [5]:
# Find the latest run directory
run_dirs = glob.glob("runs/development_run*")
if not run_dirs:
    raise FileNotFoundError("No run directories found in runs/")
latest_run_dir = Path(sorted(run_dirs)[-1])
print(f"Evaluating run at: {latest_run_dir}")

# Run evaluation on test and train sets
print("Starting Test Evaluation...")
eval_run(run_dir=latest_run_dir, period="test")

print("Starting Train Evaluation...")
eval_run(run_dir=latest_run_dir, period="train")

Evaluating run at: runs/development_run_2111_094151
Starting Test Evaluation...
2025-11-21 09:42:25,001: Using the model weights from runs/development_run_2111_094151/model_epoch030.pt


/home/sngrj0hn/GitHub/neuralhydrology/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(weight_fi

# Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]2025-11-21 09:42:25,360: Validating data availability against configured time periods...
2025-11-21 09:42:25,361:   Historical coverage: 2020-09-01 00:00:00 to 2024-04-30 23:00:00
2025-11-21 09:42:25,361:   Forecast coverage: 2020-10-01 00:00:00 to 2025-11-20 00:00:00
2025-11-21 09:42:25,361:   Checking train period: 2020-10-01 to 2023-09-30
2025-11-21 09:42:25,361:   Checking validation period: 2023-10-01 to 2024-01-31
2025-11-21 09:42:25,361:   Checking test period: 2024-02-01 to 2024-04-30
2025-11-21 09:42:25,362: ✅ Data availability validation passed - all periods fall within available data ranges
2025-11-21 09:42:25,360: Validating data availability against configured time periods...
2025-11-21 09:42:25,361:   Historical coverage: 2020-09-01 00:00:00 to 2024-04-30 23:00:00
2025-11-21 09:42:25,361:   Forecast coverage: 2020-10-01 00:00:00 to 2025-11-20 00:00:00
2025-11-21 09:42:25,361:   Checking train period: 2020-10-01 to 2023-09

/home/sngrj0hn/GitHub/neuralhydrology/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(weight_fi

# Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]2025-11-21 09:42:25,781: Validating data availability against configured time periods...
2025-11-21 09:42:25,781:   Historical coverage: 2020-09-01 00:00:00 to 2024-04-30 23:00:00
2025-11-21 09:42:25,782:   Forecast coverage: 2020-10-01 00:00:00 to 2025-11-20 00:00:00
2025-11-21 09:42:25,782:   Checking train period: 2020-10-01 to 2023-09-30
2025-11-21 09:42:25,782:   Checking validation period: 2023-10-01 to 2024-01-31
2025-11-21 09:42:25,782:   Checking test period: 2024-02-01 to 2024-04-30
2025-11-21 09:42:25,782: ✅ Data availability validation passed - all periods fall within available data ranges
2025-11-21 09:42:25,781: Validating data availability against configured time periods...
2025-11-21 09:42:25,781:   Historical coverage: 2020-09-01 00:00:00 to 2024-04-30 23:00:00
2025-11-21 09:42:25,782:   Forecast coverage: 2020-10-01 00:00:00 to 2025-11-20 00:00:00
2025-11-21 09:42:25,782:   Checking train period: 2020-10-01 to 2023-09

In [6]:
def process_results(run_dir, period, basin="DE2"):
    """Load and process results for a given period."""
    results_files = list(run_dir.glob(f"{period}/*/{period}_results.p"))
    if not results_files:
        raise FileNotFoundError(f"Could not find {period}_results.p in {run_dir}")
    
    results_file = results_files[0]
    print(f"Loading {period} results from: {results_file}")
    
    with open(results_file, "rb") as fp:
        results = pickle.load(fp)
        
    if basin not in results:
        basin = list(results.keys())[0]
        print(f"Basin {basin} not found, using: {basin}")
        
    ds = results[basin]['1h']['xr']
    
    # Process probabilistic or deterministic
    if 'samples' in ds.dims:
        print(f"Processing probabilistic {period} results...")
        quantiles = [0.05, 0.25, 0.5, 0.75, 0.95]
        ds_q = ds['discharge_vol_sim'].quantile(quantiles, dim='samples')
        df_q = ds_q.to_dataframe().reset_index()
        df_q = df_q.pivot(index=['date', 'time_step'], columns='quantile', values='discharge_vol_sim')
        df_q.columns = [f'q{int(q*100):02d}' for q in df_q.columns]
        df_q = df_q.reset_index()
        df_obs = ds['discharge_vol_obs'].to_dataframe().reset_index()
        df = pd.merge(df_q, df_obs, on=['date', 'time_step'], how='left')
        df['discharge_vol_sim'] = df['q50']
    else:
        print(f"Processing deterministic {period} results...")
        df = ds.to_dataframe().dropna().reset_index()
        
    # Calculate dates
    # date in ds is forecast_end_date
    df = df.rename(columns={'date': 'forecast_end_date'})
    
    # Use forecast horizon from config if available, else default to 240
    horizon = cfg.forecast_seq_length if 'cfg' in locals() else 240
    
    df["issue_date"] = df["forecast_end_date"] - pd.to_timedelta(horizon, unit="h")
    df["date"] = df["forecast_end_date"] + pd.to_timedelta(df["time_step"], unit="h")
    
    return df

def get_day_1_series(df):
    """Extracts the first 24 steps (Day 1) from each forecast issue."""
    df_sorted = df.sort_values(['issue_date', 'time_step'])
    day_1 = df_sorted.groupby('issue_date').head(24)
    day_1 = day_1.sort_values('date')
    return day_1

def plot_interactive_forecast(df, title):
    fig = go.Figure()
    
    # Add 90% CI
    if 'q05' in df.columns and 'q95' in df.columns:
        fig.add_trace(go.Scatter(
            x=pd.concat([df['date'], df['date'][::-1]]),
            y=pd.concat([df['q95'], df['q05'][::-1]]),
            fill='toself',
            fillcolor='rgba(0, 0, 255, 0.1)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=True,
            name='90% CI'
        ))

    # Add 50% CI
    if 'q25' in df.columns and 'q75' in df.columns:
        fig.add_trace(go.Scatter(
            x=pd.concat([df['date'], df['date'][::-1]]),
            y=pd.concat([df['q75'], df['q25'][::-1]]),
            fill='toself',
            fillcolor='rgba(0, 0, 255, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=True,
            name='50% CI'
        ))
    
    # Add Simulation Trace
    fig.add_trace(go.Scatter(
        x=df['date'], 
        y=df['discharge_vol_sim'],
        mode='lines',
        name='Forecast (Median)',
        line=dict(color='blue', width=1.5)
    ))
    
    # Add Observation Trace
    if 'discharge_vol_obs' in df.columns:
        fig.add_trace(go.Scatter(
            x=df['date'], 
            y=df['discharge_vol_obs'],
            mode='lines',
            name='Observation',
            line=dict(color='black', width=1.5, dash='solid'),
            opacity=0.7
        ))
    
    fig.update_layout(
        title=title,
        xaxis_title='Date',
        yaxis_title='Discharge Volume',
        template='plotly_white',
        hovermode='x unified',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    fig.show()

# Process and Plot Test Results
sim_test = process_results(latest_run_dir, "test")
day_1_test = get_day_1_series(sim_test)
plot_interactive_forecast(day_1_test, f'Day 1 Forecast vs Observation (Test Period)')

# Process and Plot Train Results
sim_train = process_results(latest_run_dir, "train")
day_1_train = get_day_1_series(sim_train)
plot_interactive_forecast(day_1_train, f'Day 1 Forecast vs Observation (Training Period)')

Loading test results from: runs/development_run_2111_094151/test/model_epoch030/test_results.p
Processing deterministic test results...


Loading train results from: runs/development_run_2111_094151/train/model_epoch030/train_results.p
Processing deterministic train results...
